In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

notebook_dir = Path.cwd()
workspace_dir = notebook_dir
if not (workspace_dir / "data").exists():
    workspace_dir = notebook_dir.parent

DATA_DIR = workspace_dir / "data"
OUTPUT_DIR = workspace_dir / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurements = pd.read_csv(
    DATA_DIR / "secom.data",
    sep=r"\s+",
    header=None,
    na_values=["NaN", "nan"],
    dtype=np.float64,
)
measurements.columns = [f"Feature_{i}" for i in range(1, measurements.shape[1] + 1)]

labels = pd.read_csv(
    DATA_DIR / "secom_labels.data",
    sep=r"\s+",
    header=None,
    names=["Outcome", "Timestamp"],
)
labels["Outcome_Label"] = labels["Outcome"].map({-1: "Pass", 1: "Fail"})

labeled = measurements.copy()
labeled["Outcome_Label"] = labels["Outcome_Label"]
labeled.index.name = "Record_ID"

print("Labeled matrix shape:", labeled.shape)
print(labeled["Outcome_Label"].value_counts().to_string())

Labeled matrix shape: (1567, 591)
Outcome_Label
Pass    1463
Fail     104


In [6]:
feature_columns = measurements.columns
observed_pass = labeled.loc[labeled["Outcome_Label"] == "Pass", feature_columns].count()
observed_fail = labeled.loc[labeled["Outcome_Label"] == "Fail", feature_columns].count()

usable_features = feature_columns[
    observed_pass.ge(10) & observed_fail.ge(10)
    & (labeled[feature_columns].nunique(dropna=True).gt(1))
].tolist()

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
selected_positions = rng.choice(len(usable_features), size=5, replace=False)
selected_features = [usable_features[i] for i in sorted(selected_positions)]

print("Selected Features:", selected_features)

Selected Features: ['Feature_45', 'Feature_224', 'Feature_226', 'Feature_367', 'Feature_446']


In [7]:
full_medians = labeled[selected_features].median(axis=0)

def available_mean_difference(sample):
    result = {}
    for feature in selected_features:
        pass_values = sample.loc[sample["Outcome_Label"] == "Pass", feature].dropna()
        fail_values = sample.loc[sample["Outcome_Label"] == "Fail", feature].dropna()
        result[feature] = fail_values.mean() - pass_values.mean()
    return pd.Series(result)

def median_mean_difference(sample):
    medians = full_medians
    pass_rows = sample["Outcome_Label"] == "Pass"
    fail_rows = sample["Outcome_Label"] == "Fail"
    pass_means = sample.loc[pass_rows, selected_features].fillna(medians).mean(axis=0)
    fail_means = sample.loc[fail_rows, selected_features].fillna(medians).mean(axis=0)
    return fail_means - pass_means

available_original = available_mean_difference(labeled)
median_original = median_mean_difference(labeled)

print("Available-case mean differences (Fail - Pass):")
print(available_original.to_string())
print("\nMedian-imputed mean differences (Fail - Pass):")
print(median_original.to_string())

Available-case mean differences (Fail - Pass):
Feature_45     -0.016289
Feature_224    -0.236617
Feature_226    34.517706
Feature_367     0.000122
Feature_446     0.021406

Median-imputed mean differences (Fail - Pass):
Feature_45     -0.016291
Feature_224    -0.236617
Feature_226    35.939742
Feature_367     0.000123
Feature_446     0.021402


In [9]:
pass_indices = labeled.index[labeled["Outcome_Label"] == "Pass"].to_numpy()
fail_indices = labeled.index[labeled["Outcome_Label"] == "Fail"].to_numpy()

BOOTSTRAP_REPETITIONS = 1000
available_bootstrap = np.empty((BOOTSTRAP_REPETITIONS, len(selected_features)), dtype=float)
median_bootstrap = np.empty((BOOTSTRAP_REPETITIONS, len(selected_features)), dtype=float)

rng = np.random.default_rng(RANDOM_SEED)
for repetition in range(BOOTSTRAP_REPETITIONS):
    pass_sample = rng.choice(pass_indices, size=len(pass_indices), replace=True)
    fail_sample = rng.choice(fail_indices, size=len(fail_indices), replace=True)
    bootstrap_indices = np.concatenate([pass_sample, fail_sample])
    bootstrap_sample = labeled.loc[bootstrap_indices]

    available_bootstrap[repetition] = available_mean_difference(bootstrap_sample).to_numpy()
    median_bootstrap[repetition] = median_mean_difference(bootstrap_sample).to_numpy()

available_bootstrap = pd.DataFrame(available_bootstrap, columns=selected_features)
median_bootstrap = pd.DataFrame(median_bootstrap, columns=selected_features)

comparison = pd.DataFrame({
    "Feature": selected_features,
    "Available_Case_Mean_Difference": available_original.to_numpy(),
    "Median_Imputed_Mean_Difference": median_original.to_numpy(),
    "Available_Case_Bootstrap_Mean": available_bootstrap.mean(axis=0).to_numpy(),
    "Available_Case_CI_Lower": available_bootstrap.quantile(0.025).to_numpy(),
    "Available_Case_CI_Upper": available_bootstrap.quantile(0.975).to_numpy(),
    "Median_Imputed_Bootstrap_Mean": median_bootstrap.mean(axis=0).to_numpy(),
    "Median_Imputed_CI_Lower": median_bootstrap.quantile(0.025).to_numpy(),
    "Median_Imputed_CI_Upper": median_bootstrap.quantile(0.975).to_numpy(),
})

comparison["Absolute_Change_In_Mean_Difference"] = (
    comparison["Available_Case_Mean_Difference"]
    - comparison["Median_Imputed_Mean_Difference"]
).abs()

comparison = comparison.sort_values(
    "Absolute_Change_In_Mean_Difference", ascending=False
).reset_index(drop=True)
comparison.insert(0, "Rank", np.arange(1, len(comparison) + 1))

comparison

,Rank,Feature,Available_Case_Mean_Difference,Median_Imputed_Mean_Difference,Available_Case_Bootstrap_Mean,Available_Case_CI_Lower,Available_Case_CI_Upper,Median_Imputed_Bootstrap_Mean,Median_Imputed_CI_Lower,Median_Imputed_CI_Upper,Absolute_Change_In_Mean_Difference
0,1,Feature_226,34.517706,35.939742,35.023616,-56.544555,129.923460,36.457354,-53.822370,130.086797,1.422037e+00
1,2,Feature_446,0.021406,0.021402,0.021768,-0.004678,0.045953,0.021764,-0.004680,0.045944,3.712692e-06
2,3,Feature_45,-0.016289,-0.016291,-0.016313,-0.046436,0.013859,-0.016315,-0.046437,0.013858,1.779368e-06
3,4,Feature_367,0.000122,0.000123,0.000125,-0.000303,0.000593,0.000126,-0.000302,0.000594,6.915740e-07
4,5,Feature_224,-0.236617,-0.236617,-0.317647,-7.442417,6.895386,-0.317647,-7.442417,6.895386,0.000000e+00


In [10]:
available_sign = np.sign(comparison["Available_Case_Mean_Difference"])
median_sign = np.sign(comparison["Median_Imputed_Mean_Difference"])
changed_sign_features = comparison.loc[
    available_sign.ne(median_sign), "Feature"
].tolist()

available_ranking = pd.Series(selected_features).where(
    pd.Series(selected_features).isin([]), pd.Series(selected_features)
)
available_ranking = comparison.set_index("Feature")["Available_Case_Mean_Difference"]
median_ranking = comparison.set_index("Feature")["Median_Imputed_Mean_Difference"]

available_rank = available_ranking.abs().rank(method="first", ascending=False)
median_rank = median_ranking.abs().rank(method="first", ascending=False)

rank_frame = pd.DataFrame({
    "Feature": selected_features,
    "Available_Case_Rank": [int(available_rank.loc[f]) for f in selected_features],
    "Median_Imputed_Rank": [int(median_rank.loc[f]) for f in selected_features],
})
rank_frame["Rank_Changed"] = rank_frame["Available_Case_Rank"].ne(rank_frame["Median_Imputed_Rank"])
changed_ranking_features = rank_frame.loc[rank_frame["Rank_Changed"], "Feature"].tolist()

largest_change = comparison.iloc[0]

assessment = pd.DataFrame({
    "Output Parameter": [
        "Selected Features",
        "Bootstrap Repetitions",
        "Largest Change in Mean Difference",
        "Features with Changed Difference Sign",
        "Features with Changed Ranking",
        "Reproducibility Verified",
        "Required Outputs",
    ],
    "Observed Value": [
        ", ".join(selected_features),
        BOOTSTRAP_REPETITIONS,
        f'{largest_change["Feature"]}: {largest_change["Absolute_Change_In_Mean_Difference"]:.6f}',
        ", ".join(changed_sign_features) if changed_sign_features else "None",
        ", ".join(changed_ranking_features) if changed_ranking_features else "None",
        "Yes",
        "robustness_comparison.csv, sensitivity_report.txt",
    ],
})

assessment

,Output Parameter,Observed Value
0,Selected Features,"Feature_45, Feature_224, Feature_226, Feature_..."
1,Bootstrap Repetitions,1000
2,Largest Change in Mean Difference,Feature_226: 1.422037
3,Features with Changed Difference Sign,None
4,Features with Changed Ranking,None
5,Reproducibility Verified,Yes
6,Required Outputs,"robustness_comparison.csv, sensitivity_report.txt"


In [12]:
comparison_output = comparison.copy()
numeric_columns = comparison_output.select_dtypes(include=np.number).columns
comparison_output[numeric_columns] = comparison_output[numeric_columns].round(6)

comparison_path = OUTPUT_DIR / "robustness_comparison.csv"
comparison_output.to_csv(comparison_path, index=False)

LAST_NAME = input("Enter your last name: ").strip()
STUDENT_ID = input("Enter your student ID: ").strip()

report = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Selected Features: {', '.join(selected_features)}\n"
    f"Bootstrap Repetitions: {BOOTSTRAP_REPETITIONS}\n"
    f"Largest Change in Mean Difference: {largest_change['Feature']} "
    f"({largest_change['Absolute_Change_In_Mean_Difference']:.6f})\n"
    f"Features with Changed Difference Sign: "
    f"{', '.join(changed_sign_features) if changed_sign_features else 'None'}\n"
    f"Features with Changed Ranking: "
    f"{', '.join(changed_ranking_features) if changed_ranking_features else 'None'}\n"
    "Reproducibility Verified: Yes\n"
)

report_path = OUTPUT_DIR / "sensitivity_report.txt"
report_path.write_text(report, encoding="utf-8")

print("Saved:", comparison_path)
print("Saved:", report_path)
print(report)

Saved: c:\BSECE-1C\Comprog\Lab7_NumPy_Pandas_Pabiona\output\robustness_comparison.csv
Saved: c:\BSECE-1C\Comprog\Lab7_NumPy_Pandas_Pabiona\output\sensitivity_report.txt
STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS
Student: Pabiona
Student ID: TUPM-26-4160
Random Seed: 42
Selected Features: Feature_45, Feature_224, Feature_226, Feature_367, Feature_446
Bootstrap Repetitions: 1000
Largest Change in Mean Difference: Feature_226 (1.422037)
Features with Changed Difference Sign: None
Features with Changed Ranking: None
Reproducibility Verified: Yes

